# Phase 1 — Data Cleaning and Normalization

This notebook prepares an analysis-ready enthusiast vehicle dataset for Apex Analytics.

The cleaning process will:

- preserve the original raw source data
- isolate the final project vehicle scope
- normalize manufacturer and model-family labels
- map vehicle generations
- normalize transmission descriptions
- identify and flag inconsistent source data
- inspect missing values and duplicate records
- validate price, mileage, and model-year fields
- export a cleaned dataset for PostgreSQL and downstream analysis

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

In [4]:
DATA_PATH = Path("../data/raw/cars.csv")
PROCESSED_PATH = Path("../data/processed")

print("Dataset exists:", DATA_PATH.exists())

Dataset exists: True


In [5]:
df_raw = pd.read_csv(DATA_PATH)

print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])

Rows: 762091
Columns: 20


## 1. Create a Traceable Working Copy

The raw dataset is preserved unchanged. A separate working DataFrame is created for all cleaning and normalization steps.

A `source_row_id` field is added so that any cleaned record can be traced back to its original row in the source dataset.

In [6]:
df = df_raw.copy()

df.insert(
    0,
    "source_row_id",
    df.index
)

print("Raw rows:", len(df_raw))
print("Working rows:", len(df))
print("Working columns:", df.shape[1])

Raw rows: 762091
Working rows: 762091
Working columns: 21


In [7]:
df[
    [
        "source_row_id",
        "manufacturer",
        "model",
        "year",
        "mileage",
        "price"
    ]
].head()

,source_row_id,manufacturer,model,year,mileage,price
0,0,Acura,ILX Hybrid 1.5L,2013,92945.0,13988.0
1,1,Acura,ILX Hybrid 1.5L,2013,47645.0,17995.0
2,2,Acura,ILX Hybrid 1.5L,2013,53422.0,17000.0
3,3,Acura,ILX Hybrid 1.5L,2013,117598.0,14958.0
4,4,Acura,ILX Hybrid 1.5L,2013,114865.0,14498.0


In [8]:
project_brands = [
    "BMW",
    "Mercedes-Benz",
    "Porsche",
    "Audi",
    "Chevrolet",
    "Toyota",
    "Nissan",
    "Cadillac"
]

In [9]:
df_project_brands = df[
    df["manufacturer"].isin(project_brands)
].copy()

print("Rows in full raw dataset:", len(df))
print("Rows from project brands:", len(df_project_brands))

Rows in full raw dataset: 762091
Rows from project brands: 289619


In [10]:
df_project_brands["manufacturer"].value_counts()

manufacturer
Toyota           59535
Chevrolet        56043
Nissan           48529
Mercedes-Benz    40824
BMW              37570
Audi             17863
Cadillac         17794
Porsche          11461
Name: count, dtype: int64

## 2. Normalize Enthusiast Model Families

Marketplace model names are inconsistent and often include trim, drivetrain, or package information.

A standardized `model_family` field is created so that listings referring to the same enthusiast vehicle can be analyzed together while preserving the original `model` field.

In [11]:
df_project_brands["model_family"] = pd.NA
df_project_brands["project_scope"] = pd.NA

In [12]:
df_project_brands[
    ["manufacturer", "model", "model_family", "project_scope"]
].head()

,manufacturer,model,model_family,project_scope
8489,Audi,A8 e L 60,<NA>,<NA>
8490,Audi,A8 e L 60,<NA>,<NA>
8491,Audi,A8 e L 60,<NA>,<NA>
8492,Audi,A8 e L 60,<NA>,<NA>
8493,Audi,A8 e L 60,<NA>,<NA>


In [13]:
bmw_mask = df_project_brands["manufacturer"] == "BMW"

bmw_model_text = (
    df_project_brands.loc[bmw_mask, "model"]
    .str.upper()
)

In [14]:
for model in ["M2", "M3", "M4", "M5"]:
    
    mask = (
        bmw_mask &
        df_project_brands["model"]
        .str.contains(
            rf"\b{model}\b",
            case=False,
            na=False,
            regex=True
        )
    )

    df_project_brands.loc[
        mask,
        "model_family"
    ] = model

In [15]:
df_project_brands.loc[
    bmw_mask &
    df_project_brands["model_family"].isin(["M3", "M4"]),
    "project_scope"
] = "Core"

df_project_brands.loc[
    bmw_mask &
    df_project_brands["model_family"].isin(["M2", "M5"]),
    "project_scope"
] = "Supporting"

In [16]:
bmw_clean_check = (
    df_project_brands.loc[
        bmw_mask &
        df_project_brands["model_family"].notna()
    ]
    .groupby(
        ["model_family", "project_scope"]
    )
    .size()
)

bmw_clean_check

model_family  project_scope
M2            Supporting       157
M3            Core             528
M4            Core             565
M5            Supporting       396
dtype: int64

In [17]:
df_project_brands.loc[
    bmw_mask &
    df_project_brands["model_family"].notna(),
    ["model", "model_family"]
].drop_duplicates().sort_values(
    ["model_family", "model"]
).head(100)

,model,model_family
43671,M2,M2
43651,M2 Base,M2
43650,M2 CS,M2
43803,M2 CS Coupe,M2
43789,M2 CS Racing Cup,M2
43649,M2 Competition,M2
43734,M2 Competition Coupe,M2
41571,M3,M3
41570,M3 Base,M3
41662,M3 Base (M6),M3


### Mercedes-AMG Model Normalization

Mercedes-Benz performance listings use several inconsistent naming conventions. C63, E63, AMG GT sports car, and AMG GT 4-Door listings are normalized into standardized model families while preserving the original marketplace model description.

In [18]:
mercedes_mask = (
    df_project_brands["manufacturer"] == "Mercedes-Benz"
)

mercedes_model_text = (
    df_project_brands["model"]
    .str.upper()
)

In [19]:
c63_mask = (
    mercedes_mask &
    mercedes_model_text.str.contains(
        r"\bC\s*63\b",
        regex=True,
        na=False
    )
)

df_project_brands.loc[
    c63_mask,
    "model_family"
] = "C63"

In [20]:
e63_mask = (
    mercedes_mask &
    mercedes_model_text.str.contains(
        r"\bE\s*63\b",
        regex=True,
        na=False
    )
)

df_project_brands.loc[
    e63_mask,
    "model_family"
] = "E63"

In [21]:
amg_gt_4door_mask = (
    mercedes_mask &
    mercedes_model_text.str.contains(
        r"\bAMG\s+GT\s+(?:43|53|63)\b",
        regex=True,
        na=False
    )
)

df_project_brands.loc[
    amg_gt_4door_mask,
    "model_family"
] = "AMG GT 4-Door"

In [22]:
amg_gt_sports_mask = (
    mercedes_mask &
    mercedes_model_text.str.contains(
        r"\bAMG\s+GT\b",
        regex=True,
        na=False
    )
    & ~amg_gt_4door_mask
    & ~mercedes_model_text.str.contains(
        r"\bSLS\b",
        regex=True,
        na=False
    )
)

df_project_brands.loc[
    amg_gt_sports_mask,
    "model_family"
] = "AMG GT Sports Car"

In [23]:
df_project_brands.loc[
    mercedes_mask &
    (df_project_brands["model_family"] == "C63"),
    "project_scope"
] = "Core"

df_project_brands.loc[
    mercedes_mask &
    df_project_brands["model_family"].isin([
        "E63",
        "AMG GT Sports Car",
        "AMG GT 4-Door"
    ]),
    "project_scope"
] = "Supporting"

In [24]:
mercedes_clean_check = (
    df_project_brands.loc[
        mercedes_mask &
        df_project_brands["model_family"].notna()
    ]
    .groupby(
        ["model_family", "project_scope"]
    )
    .size()
)

mercedes_clean_check

model_family       project_scope
AMG GT 4-Door      Supporting       314
AMG GT Sports Car  Supporting       297
C63                Core             342
E63                Supporting       220
dtype: int64

### Porsche Model Normalization

Porsche listings contain many trim-level descriptions. For the cleaning phase, these are grouped into standardized model families for 911, Cayman / 718 Coupe, and Boxster / 718 Roadster.

In [25]:
porsche_mask = (
    df_project_brands["manufacturer"] == "Porsche"
)

porsche_model_text = (
    df_project_brands["model"]
    .str.upper()
)

In [26]:
porsche_911_mask = (
    porsche_mask &
    porsche_model_text.str.contains(
        r"\b911\b",
        regex=True,
        na=False
    )
)

df_project_brands.loc[
    porsche_911_mask,
    "model_family"
] = "911"

In [27]:
porsche_cayman_mask = (
    porsche_mask &
    porsche_model_text.str.contains(
        r"\bCAYMAN\b",
        regex=True,
        na=False
    )
)

df_project_brands.loc[
    porsche_cayman_mask,
    "model_family"
] = "Cayman / 718 Coupe"

In [28]:
porsche_boxster_mask = (
    porsche_mask &
    (
        porsche_model_text.str.contains(
            r"\bBOXSTER\b",
            regex=True,
            na=False
        )
        |
        porsche_model_text.str.contains(
            r"\b718\s+SPYDER\b",
            regex=True,
            na=False
        )
    )
)

df_project_brands.loc[
    porsche_boxster_mask,
    "model_family"
] = "Boxster / 718 Roadster"

In [29]:
df_project_brands.loc[
    porsche_mask &
    df_project_brands["model_family"].isin([
        "911",
        "Cayman / 718 Coupe"
    ]),
    "project_scope"
] = "Core"

df_project_brands.loc[
    porsche_mask &
    (
        df_project_brands["model_family"]
        == "Boxster / 718 Roadster"
    ),
    "project_scope"
] = "Supporting"

In [30]:
porsche_clean_check = (
    df_project_brands.loc[
        porsche_mask &
        df_project_brands["model_family"].notna()
    ]
    .groupby(
        ["model_family", "project_scope"]
    )
    .size()
)

porsche_clean_check

model_family            project_scope
911                     Core             3023
Boxster / 718 Roadster  Supporting        627
Cayman / 718 Coupe      Core              427
dtype: int64

### Audi RS Model Normalization

Audi RS listings use several naming conventions and trim descriptions. RS3, RS5, and RS7 are retained for Apex Analytics, while RS6 is excluded because the dataset contains too few observations for reliable analysis.

In [31]:
audi_mask = (
    df_project_brands["manufacturer"] == "Audi"
)

audi_model_text = (
    df_project_brands["model"]
    .str.upper()
)

In [32]:
audi_rs3_mask = (
    audi_mask &
    audi_model_text.str.contains(
        r"\bRS\s*3\b",
        regex=True,
        na=False
    )
)

df_project_brands.loc[
    audi_rs3_mask,
    "model_family"
] = "RS3"

In [33]:
audi_rs5_mask = (
    audi_mask &
    audi_model_text.str.contains(
        r"\bRS\s*5\b",
        regex=True,
        na=False
    )
)

df_project_brands.loc[
    audi_rs5_mask,
    "model_family"
] = "RS5"

In [34]:
audi_rs7_mask = (
    audi_mask &
    audi_model_text.str.contains(
        r"\bRS\s*7\b",
        regex=True,
        na=False
    )
)

df_project_brands.loc[
    audi_rs7_mask,
    "model_family"
] = "RS7"

In [35]:
audi_rs6_mask = (
    audi_mask &
    audi_model_text.str.contains(
        r"\bRS\s*6\b",
        regex=True,
        na=False
    )
)

df_project_brands.loc[
    audi_rs6_mask,
    "model_family"
] = "RS6"

df_project_brands.loc[
    audi_rs6_mask,
    "project_scope"
] = "Exclude"

In [36]:
df_project_brands.loc[
    audi_mask &
    (df_project_brands["model_family"] == "RS5"),
    "project_scope"
] = "Core"

df_project_brands.loc[
    audi_mask &
    df_project_brands["model_family"].isin([
        "RS3",
        "RS7"
    ]),
    "project_scope"
] = "Supporting"

In [37]:
audi_clean_check = (
    df_project_brands.loc[
        audi_mask &
        df_project_brands["model_family"].notna()
    ]
    .groupby(
        ["model_family", "project_scope"]
    )
    .size()
)

audi_clean_check

model_family  project_scope
RS3           Supporting        58
RS5           Core             208
RS6           Exclude            6
RS7           Supporting       137
dtype: int64

### Chevrolet Corvette Generation Normalization

Corvette listings span multiple generations with very different market behavior. Listings are grouped into C5, C6, C7, and C8 generations so pricing and mileage comparisons remain meaningful. Earlier generations are excluded from the core project scope.

In [38]:
chevrolet_mask = (
    df_project_brands["manufacturer"] == "Chevrolet"
)

corvette_mask = (
    chevrolet_mask &
    df_project_brands["model"]
    .str.contains(
        r"\bCorvette\b",
        case=False,
        na=False,
        regex=True
    )
)

In [39]:
df_project_brands["generation"] = pd.NA

In [40]:
df_project_brands.loc[
    corvette_mask &
    df_project_brands["year"].between(1997, 2004),
    "generation"
] = "C5"

df_project_brands.loc[
    corvette_mask &
    df_project_brands["year"].between(2005, 2013),
    "generation"
] = "C6"

df_project_brands.loc[
    corvette_mask &
    df_project_brands["year"].between(2014, 2019),
    "generation"
] = "C7"

df_project_brands.loc[
    corvette_mask &
    (df_project_brands["year"] >= 2020),
    "generation"
] = "C8"

In [41]:
for generation in ["C5", "C6", "C7", "C8"]:
    mask = (
        corvette_mask &
        (df_project_brands["generation"] == generation)
    )

    df_project_brands.loc[
        mask,
        "model_family"
    ] = f"Corvette {generation}"

In [42]:
df_project_brands.loc[
    corvette_mask &
    df_project_brands["generation"].isin(["C6", "C7", "C8"]),
    "project_scope"
] = "Core"

df_project_brands.loc[
    corvette_mask &
    (df_project_brands["generation"] == "C5"),
    "project_scope"
] = "Supporting"

In [43]:
corvette_clean_check = (
    df_project_brands.loc[
        corvette_mask &
        df_project_brands["model_family"].notna()
    ]
    .groupby(
        ["model_family", "project_scope"]
    )
    .size()
)

corvette_clean_check

model_family  project_scope
Corvette C5   Supporting        287
Corvette C6   Core              532
Corvette C7   Core             1006
Corvette C8   Core              943
dtype: int64

### Toyota Supra Generation Normalization

The dataset contains sparse observations for older Supra generations but strong coverage for the modern A90/A91 generation. Apex Analytics therefore retains 2020–2023 Supra listings as the primary modern Supra segment.

In [44]:
toyota_mask = (
    df_project_brands["manufacturer"] == "Toyota"
)

supra_mask = (
    toyota_mask &
    df_project_brands["model"]
    .str.contains(
        r"\bSupra\b",
        case=False,
        na=False,
        regex=True
    )
)

In [45]:
df_project_brands.loc[
    supra_mask &
    (df_project_brands["year"] >= 2020),
    "generation"
] = "A90/A91"

In [46]:
df_project_brands.loc[
    supra_mask &
    (df_project_brands["generation"] == "A90/A91"),
    "model_family"
] = "Supra A90/A91"

In [47]:
df_project_brands.loc[
    supra_mask &
    (df_project_brands["model_family"] == "Supra A90/A91"),
    "project_scope"
] = "Core"

In [48]:
supra_clean_check = (
    df_project_brands.loc[
        supra_mask &
        df_project_brands["model_family"].notna()
    ]
    .groupby(
        ["model_family", "project_scope"]
    )
    .size()
)

supra_clean_check

model_family   project_scope
Supra A90/A91  Core             271
dtype: int64

### Nissan GT-R Model Normalization

Nissan GT-R listings use several trim-level descriptions. Because the available records all represent the R35 generation, they are normalized into a single GT-R (R35) model family for supporting market analysis.

In [49]:
nissan_mask = (
    df_project_brands["manufacturer"] == "Nissan"
)

gtr_mask = (
    nissan_mask &
    df_project_brands["model"]
    .str.contains(
        r"\bGT[\s-]?R\b",
        case=False,
        na=False,
        regex=True
    )
)

In [50]:
df_project_brands.loc[
    gtr_mask,
    "model_family"
] = "GT-R (R35)"

In [51]:
df_project_brands.loc[
    gtr_mask,
    "generation"
] = "R35"

In [52]:
df_project_brands.loc[
    gtr_mask,
    "project_scope"
] = "Supporting"

In [53]:
gtr_clean_check = (
    df_project_brands.loc[
        gtr_mask &
        df_project_brands["model_family"].notna()
    ]
    .groupby(
        ["model_family", "project_scope"]
    )
    .size()
)

gtr_clean_check

model_family  project_scope
GT-R (R35)    Supporting       154
dtype: int64

In [54]:
df_project_brands.loc[
    gtr_mask,
    ["year", "model_family", "generation"]
].groupby(
    ["model_family", "generation"]
).agg(
    listings=("year", "size"),
    earliest_year=("year", "min"),
    latest_year=("year", "max")
)

,,listings,earliest_year,latest_year
model_family,generation,,,
GT-R (R35),R35,154,2009,2023


### Cadillac Blackwing Model Normalization

Cadillac performance listings use multiple naming conventions for CT4-V and CT5-V Blackwing models. These are normalized into two supporting enthusiast model families while preserving the original listing descriptions.

In [55]:
cadillac_mask = (
    df_project_brands["manufacturer"] == "Cadillac"
)

cadillac_model_text = (
    df_project_brands["model"]
    .str.upper()
)

In [56]:
ct4_blackwing_mask = (
    cadillac_mask &
    cadillac_model_text.str.contains(
        r"\bCT4[\s-]*V\b.*\bBLACKWING\b",
        regex=True,
        na=False
    )
)

df_project_brands.loc[
    ct4_blackwing_mask,
    "model_family"
] = "CT4-V Blackwing"

In [57]:
ct5_blackwing_mask = (
    cadillac_mask &
    cadillac_model_text.str.contains(
        r"\bCT5[\s-]*V\b.*\bBLACKWING\b",
        regex=True,
        na=False
    )
)

df_project_brands.loc[
    ct5_blackwing_mask,
    "model_family"
] = "CT5-V Blackwing"

In [58]:
df_project_brands.loc[
    cadillac_mask &
    df_project_brands["model_family"].isin([
        "CT4-V Blackwing",
        "CT5-V Blackwing"
    ]),
    "project_scope"
] = "Supporting"

In [59]:
cadillac_clean_check = (
    df_project_brands.loc[
        cadillac_mask &
        df_project_brands["model_family"].notna()
    ]
    .groupby(
        ["model_family", "project_scope"]
    )
    .size()
)

cadillac_clean_check

model_family     project_scope
CT4-V Blackwing  Supporting       81
CT5-V Blackwing  Supporting       57
dtype: int64

## 3. Create the Enthusiast-Only Dataset

After normalizing model families and project scope, the dataset is filtered to retain only vehicles classified as Core or Supporting.

Rows marked Exclude or rows that do not belong to an approved enthusiast model family are removed from the working analysis dataset.

In [60]:
enthusiast_clean = (
    df_project_brands[
        df_project_brands["project_scope"].isin([
            "Core",
            "Supporting"
        ])
    ]
    .copy()
)

In [61]:
print("Included enthusiast listings:", len(enthusiast_clean))

Included enthusiast listings: 10630


In [62]:
enthusiast_clean["project_scope"].value_counts()

project_scope
Core          7845
Supporting    2785
Name: count, dtype: int64

In [63]:
model_reconciliation = (
    enthusiast_clean
    .groupby(
        ["manufacturer", "model_family", "project_scope"]
    )
    .size()
    .reset_index(name="listings")
    .sort_values(
        ["project_scope", "listings"],
        ascending=[True, False]
    )
)

model_reconciliation

,manufacturer,model_family,project_scope,listings
18,Porsche,911,Core,3023
11,Chevrolet,Corvette C7,Core,1006
12,Chevrolet,Corvette C8,Core,943
5,BMW,M4,Core,565
10,Chevrolet,Corvette C6,Core,532
4,BMW,M3,Core,528
20,Porsche,Cayman / 718 Coupe,Core,427
15,Mercedes-Benz,C63,Core,342
21,Toyota,Supra A90/A91,Core,271
1,Audi,RS5,Core,208


In [64]:
print(
    "Missing model_family:",
    enthusiast_clean["model_family"].isna().sum()
)

print(
    "Missing project_scope:",
    enthusiast_clean["project_scope"].isna().sum()
)

print(
    "Excluded rows remaining:",
    (enthusiast_clean["project_scope"] == "Exclude").sum()
)

Missing model_family: 0
Missing project_scope: 0
Excluded rows remaining: 0


In [65]:
enthusiast_clean["manufacturer"].value_counts()

manufacturer
Porsche          4077
Chevrolet        2768
BMW              1646
Mercedes-Benz    1173
Audi              403
Toyota            271
Nissan            154
Cadillac          138
Name: count, dtype: int64

In [66]:
print("Rows:", enthusiast_clean.shape[0])
print("Columns:", enthusiast_clean.shape[1])

Rows: 10630
Columns: 24


## 4. Transmission Normalization

The raw marketplace dataset contains inconsistent transmission descriptions, including multiple labels for the same transmission type and some values that conflict with known vehicle configurations.

The original `transmission` field will be preserved. A separate normalized transmission field and data-quality flag will be created for downstream analysis.

In [67]:
transmission_counts = (
    enthusiast_clean["transmission"]
    .value_counts(dropna=False)
    .rename_axis("raw_transmission")
    .reset_index(name="listings")
)

transmission_counts.head(60)

,raw_transmission,listings
0,7-Speed Automatic with Auto-Shift,1997
1,8-Speed Automatic,1380
2,6-Speed Manual,1198
3,Automatic,1160
4,8-Speed Automatic with Auto-Shift,824
5,9-Speed Automatic,473
6,Manual,416
7,NaN,323
8,6-Speed Automatic,308
9,7-Speed Automatic,263


In [68]:
print(
    "Unique transmission descriptions:",
    enthusiast_clean["transmission"].nunique(dropna=True)
)

print(
    "Missing transmission values:",
    enthusiast_clean["transmission"].isna().sum()
)

Unique transmission descriptions: 135
Missing transmission values: 323


In [69]:
transmission_counts.tail(60)

,raw_transmission,listings
76,"Manual, 7-Spd",2
77,5-Speed Manual,2
78,5-Speed Automatic with Tiptronic S,2
79,"Auto, 7-Spd S trnc Spt",1
80,8-Speed Automatic Steptronic,1
81,Auto 7Spd DCT w/Drivelogic,1
82,5-SPEED M/T,1
83,SMG,1
84,6-Speed DSG Automatic with Tiptronic,1
85,"Manual, 6-Spd w/Overdrive",1


In [70]:
transmission_by_model = (
    enthusiast_clean
    .groupby(["model_family", "transmission"])
    .size()
    .reset_index(name="listings")
    .sort_values(
        ["model_family", "listings"],
        ascending=[True, False]
    )
)

transmission_by_model.head(100)

,model_family,transmission,listings
27,911,7-Speed Automatic with Auto-Shift,867
37,911,8-Speed Automatic with Auto-Shift,476
18,911,6-Speed Manual,426
40,911,Automatic,339
48,911,Manual,177
...,...,...,...
77,Boxster / 718 Roadster,5 Speed,1
78,Boxster / 718 Roadster,5 speed Manual,1
79,Boxster / 718 Roadster,5 speed manual,1
80,Boxster / 718 Roadster,5-Spd Manual,1


In [71]:
transmission_by_model[
    transmission_by_model["model_family"] == "Corvette C8"
]

,model_family,transmission,listings
218,Corvette C8,8-Speed Automatic with Auto-Shift,348
222,Corvette C8,Automatic,229
215,Corvette C8,8-Speed,220
217,Corvette C8,8-Speed Automatic,37
216,Corvette C8,8-Speed A/T,31
226,Corvette C8,Transmission w/Dual Shift Mode,22
220,Corvette C8,8-Speed Manual,13
221,Corvette C8,A/T,4
219,Corvette C8,8-Speed Double Clutch,3
224,Corvette C8,Manual,2


In [72]:
transmission_by_model[
    transmission_by_model["model_family"] == "M3"
].head(30)

,model_family,transmission,listings
269,M3,6-Speed Manual,145
271,M3,7-Speed Automatic with Auto-Shift,129
274,M3,8-Speed Automatic,59
281,M3,Automatic,36
286,M3,Manual,33
266,M3,6-Speed Automatic with Auto-Shift,19
262,M3,5-Speed Manual,15
268,M3,6-Speed M/T,13
276,M3,8-Speed Automatic with Steptronic,9
292,M3,Transmission w/Dual Shift Mode,9


### Transmission Classification Strategy

Raw transmission descriptions are preserved unchanged.

Two normalized fields are created:

- `transmission_type`: distinguishes Manual, Automatic, Dual-Clutch, and Unknown/Ambiguous transmissions.
- `transmission_group`: provides a simplified Manual vs Automatic grouping for statistical analysis.

Ambiguous or contradictory source records are not force-corrected. Instead, they are flagged for exclusion from transmission-specific analyses.

In [73]:
def normalize_transmission(value):
    if pd.isna(value):
        return "Unknown/Ambiguous"

    text = str(value).strip().upper()

    # Explicitly unknown
    if text in ["NOT SPECIFIED", "UNKNOWN", "N/A", ""]:
        return "Unknown/Ambiguous"

    # Dual-clutch / PDK
    dual_clutch_terms = [
        "DUAL CLUTCH",
        "DOUBLE CLUTCH",
        "DOPPELKUPPLUNG",
        "PDK"
    ]

    if any(term in text for term in dual_clutch_terms):
        return "Dual-Clutch"

    # Automatic-family descriptions
    automatic_terms = [
        "AUTOMATIC",
        "A/T",
        "AUTO-SHIFT",
        "TIPTRONIC",
        "STEPTRONIC",
        "CVT",
        "DUAL SHIFT MODE",
        "AUTOMATIC MODES"
    ]

    if any(term in text for term in automatic_terms):
        return "Automatic"

    # Manual descriptions
    manual_terms = [
        "MANUAL",
        "M/T"
    ]

    if any(term in text for term in manual_terms):
        return "Manual"

    # Labels such as "8-Speed" do not tell us enough
    return "Unknown/Ambiguous"

In [74]:
enthusiast_clean["transmission_type"] = (
    enthusiast_clean["transmission"]
    .apply(normalize_transmission)
)

In [75]:
transmission_group_map = {
    "Manual": "Manual",
    "Automatic": "Automatic",
    "Dual-Clutch": "Automatic",
    "Unknown/Ambiguous": "Unknown"
}

enthusiast_clean["transmission_group"] = (
    enthusiast_clean["transmission_type"]
    .map(transmission_group_map)
)

In [76]:
enthusiast_clean["transmission_type"].value_counts()

transmission_type
Automatic            7572
Manual               2265
Unknown/Ambiguous     733
Dual-Clutch            60
Name: count, dtype: int64

In [77]:
enthusiast_clean["transmission_group"].value_counts()

transmission_group
Automatic    7632
Manual       2265
Unknown       733
Name: count, dtype: int64

In [78]:
enthusiast_clean["transmission_quality_flag"] = "Parsed"

# Ambiguous descriptions that actually contain a value
enthusiast_clean.loc[
    enthusiast_clean["transmission"].notna() &
    (enthusiast_clean["transmission_type"] == "Unknown/Ambiguous"),
    "transmission_quality_flag"
] = "Ambiguous"

# Truly missing source values
enthusiast_clean.loc[
    enthusiast_clean["transmission"].isna(),
    "transmission_quality_flag"
] = "Missing"

# Known C8 source inconsistencies
enthusiast_clean.loc[
    c8_manual_conflict,
    "transmission_quality_flag"
] = "Source inconsistency"

NameError: name 'c8_manual_conflict' is not defined

In [ ]:
c8_manual_conflict = (
    (enthusiast_clean["model_family"] == "Corvette C8") &
    (enthusiast_clean["transmission_type"] == "Manual")
)

print(
    "C8 manual-label conflicts:",
    c8_manual_conflict.sum()
)

In [ ]:
enthusiast_clean.loc[
    c8_manual_conflict,
    "transmission_type"
] = "Unknown/Ambiguous"

enthusiast_clean.loc[
    c8_manual_conflict,
    "transmission_group"
] = "Unknown"

enthusiast_clean.loc[
    c8_manual_conflict,
    "transmission_quality_flag"
] = "Source inconsistency"

In [ ]:
pd.crosstab(
    enthusiast_clean.loc[
        enthusiast_clean["model_family"] == "Corvette C8",
        "transmission_type"
    ],
    columns="listings"
)

In [ ]:
enthusiast_clean.loc[
    enthusiast_clean["model_family"] == "Corvette C8",
    "transmission_type"
].value_counts()

In [ ]:
enthusiast_clean[
    "transmission_quality_flag"
].value_counts()

In [ ]:
pd.crosstab(
    enthusiast_clean["model_family"],
    enthusiast_clean["transmission_group"],
    margins=True
)

## 5. Price and Mileage Validation

Price and mileage are central to the downstream market analysis. Both fields are inspected for missing values, impossible values, and extreme observations before analytical use.

Potential outliers are identified and documented rather than automatically removed.

In [ ]:
enthusiast_clean[
    ["price", "mileage"]
].describe(
    percentiles=[
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99
    ]
)

In [ ]:
print(
    "Missing price:",
    enthusiast_clean["price"].isna().sum()
)

print(
    "Missing mileage:",
    enthusiast_clean["mileage"].isna().sum()
)

print(
    "Price <= 0:",
    (enthusiast_clean["price"] <= 0).sum()
)

print(
    "Mileage < 0:",
    (enthusiast_clean["mileage"] < 0).sum()
)

In [ ]:
enthusiast_clean.nlargest(
    20,
    "price"
)[
    [
        "manufacturer",
        "model_family",
        "year",
        "model",
        "price",
        "mileage"
    ]
]

In [ ]:
enthusiast_clean.nlargest(
    20,
    "mileage"
)[
    [
        "manufacturer",
        "model_family",
        "year",
        "model",
        "price",
        "mileage"
    ]
]

In [ ]:
enthusiast_clean.loc[
    enthusiast_clean["mileage"].isna(),
    [
        "source_row_id",
        "manufacturer",
        "model_family",
        "year",
        "model",
        "price",
        "mileage"
    ]
]

In [ ]:
enthusiast_clean["mileage_quality_flag"] = "Valid"

enthusiast_clean.loc[
    enthusiast_clean["mileage"].isna(),
    "mileage_quality_flag"
] = "Missing"

## 6. Duplicate Record Audit

Potential duplicate marketplace listings are identified before downstream analysis.

Because the dataset does not provide a unique vehicle identifier such as VIN, duplicate detection is performed conservatively. Records are first compared across the original source fields rather than being removed solely because multiple vehicles share similar characteristics.

In [ ]:
raw_columns = df_raw.columns.tolist()

raw_columns

In [ ]:
exact_duplicate_mask = enthusiast_clean.duplicated(
    subset=raw_columns,
    keep=False
)

print(
    "Rows involved in exact duplicate groups:",
    exact_duplicate_mask.sum()
)

In [ ]:
exact_duplicates = (
    enthusiast_clean.loc[
        exact_duplicate_mask,
        ["source_row_id"] + raw_columns
    ]
    .sort_values(raw_columns)
)

exact_duplicates.head(30)

In [ ]:
duplicate_excess = enthusiast_clean.duplicated(
    subset=raw_columns,
    keep="first"
).sum()

print(
    "Exact duplicate rows beyond first occurrence:",
    duplicate_excess
)

In [ ]:
removed_exact_duplicates = enthusiast_clean.loc[
    enthusiast_clean.duplicated(
        subset=raw_columns,
        keep="first"
    )
].copy()

print(
    "Rows marked for removal:",
    len(removed_exact_duplicates)
)

In [ ]:
rows_before_dedup = len(enthusiast_clean)

enthusiast_clean = (
    enthusiast_clean
    .drop_duplicates(
        subset=raw_columns,
        keep="first"
    )
    .copy()
)

rows_after_dedup = len(enthusiast_clean)

print("Rows before deduplication:", rows_before_dedup)
print("Rows after deduplication:", rows_after_dedup)
print("Rows removed:", rows_before_dedup - rows_after_dedup)

In [ ]:
remaining_duplicates = enthusiast_clean.duplicated(
    subset=raw_columns,
    keep=False
).sum()

print(
    "Exact duplicate rows remaining:",
    remaining_duplicates
)

### Duplicate Cleaning Result

The audit identified 300 rows belonging to exact duplicate groups. Because 150 rows were redundant copies beyond the first occurrence, 150 exact duplicate records were removed.

Duplicate identification required all original source fields to match exactly. Similar vehicles were not treated as duplicates unless every original field matched, minimizing the risk of removing legitimate separate listings.

The cleaned dataset decreased from 10,630 to 10,480 rows.

In [ ]:
enthusiast_clean["project_scope"].value_counts()

## 7. Vehicle Generation Mapping

Model-year ranges are used to map enthusiast vehicles to platform generations.

Generation mapping is necessary because vehicles from different generations can have fundamentally different engines, transmissions, technology, performance characteristics, and market behavior.

Generation labels are added separately from the normalized `model_family` field so the original marketplace model description remains preserved.

In [79]:
m3_mask = (
    (enthusiast_clean["manufacturer"] == "BMW") &
    (enthusiast_clean["model_family"] == "M3")
)

m3_generation_map = [
    (1988, 1991, "E30"),
    (1995, 1999, "E36"),
    (2001, 2006, "E46"),
    (2008, 2013, "E9x"),
    (2015, 2018, "F80"),
    (2021, 2023, "G80")
]

for start_year, end_year, generation in m3_generation_map:
    enthusiast_clean.loc[
        m3_mask &
        enthusiast_clean["year"].between(start_year, end_year),
        "generation"
    ] = generation

In [80]:
m4_mask = (
    (enthusiast_clean["manufacturer"] == "BMW") &
    (enthusiast_clean["model_family"] == "M4")
)

enthusiast_clean.loc[
    m4_mask &
    enthusiast_clean["year"].between(2015, 2020),
    "generation"
] = "F82/F83"

enthusiast_clean.loc[
    m4_mask &
    enthusiast_clean["year"].between(2021, 2023),
    "generation"
] = "G82/G83"

In [81]:
m2_mask = (
    (enthusiast_clean["manufacturer"] == "BMW") &
    (enthusiast_clean["model_family"] == "M2")
)

enthusiast_clean.loc[
    m2_mask &
    enthusiast_clean["year"].between(2016, 2021),
    "generation"
] = "F87"

In [82]:
m5_mask = (
    (enthusiast_clean["manufacturer"] == "BMW") &
    (enthusiast_clean["model_family"] == "M5")
)

m5_generation_map = [
    (1988, 1988, "E28"),
    (1991, 1993, "E34"),
    (2000, 2003, "E39"),
    (2006, 2010, "E60"),
    (2013, 2016, "F10"),
    (2018, 2023, "F90")
]

for start_year, end_year, generation in m5_generation_map:
    enthusiast_clean.loc[
        m5_mask &
        enthusiast_clean["year"].between(start_year, end_year),
        "generation"
    ] = generation

In [83]:
bmw_generation_check = (
    enthusiast_clean.loc[
        enthusiast_clean["manufacturer"] == "BMW"
    ]
    .groupby(
        ["model_family", "generation"],
        dropna=False
    )
    .size()
    .reset_index(name="listings")
)

bmw_generation_check

,model_family,generation,listings
0,M2,F87,157
1,M3,E30,5
2,M3,E36,32
3,M3,E46,87
4,M3,E9x,119
5,M3,F80,146
6,M3,G80,139
7,M4,F82/F83,349
8,M4,G82/G83,216
9,M5,E28,3


In [84]:
bmw_unmapped = enthusiast_clean.loc[
    (enthusiast_clean["manufacturer"] == "BMW") &
    (enthusiast_clean["generation"].isna()),
    [
        "source_row_id",
        "model_family",
        "year",
        "model"
    ]
]

print("Unmapped BMW rows:", len(bmw_unmapped))

bmw_unmapped.sort_values(
    ["model_family", "year"]
).head(50)

Unmapped BMW rows: 0


,source_row_id,model_family,year,model


### Mercedes-AMG Generation Mapping

Mercedes-AMG vehicles are mapped to platform generations using model year and normalized model family.

Generation mapping allows comparisons such as W204-era vs. W205-era C63 pricing while preventing fundamentally different vehicle generations from being treated as equivalent observations.

In [85]:
c63_mask = (
    (enthusiast_clean["manufacturer"] == "Mercedes-Benz") &
    (enthusiast_clean["model_family"] == "C63")
)

enthusiast_clean.loc[
    c63_mask &
    enthusiast_clean["year"].between(2008, 2014),
    "generation"
] = "W204/C204"

enthusiast_clean.loc[
    c63_mask &
    enthusiast_clean["year"].between(2015, 2021),
    "generation"
] = "W205/C205"

In [86]:
e63_mask = (
    (enthusiast_clean["manufacturer"] == "Mercedes-Benz") &
    (enthusiast_clean["model_family"] == "E63")
)

enthusiast_clean.loc[
    e63_mask &
    enthusiast_clean["year"].between(2007, 2009),
    "generation"
] = "W211"

enthusiast_clean.loc[
    e63_mask &
    enthusiast_clean["year"].between(2010, 2016),
    "generation"
] = "W212/S212"

enthusiast_clean.loc[
    e63_mask &
    enthusiast_clean["year"].between(2017, 2021),
    "generation"
] = "W213/S213"

In [87]:
amg_gt_sports_mask = (
    (enthusiast_clean["manufacturer"] == "Mercedes-Benz") &
    (enthusiast_clean["model_family"] == "AMG GT Sports Car")
)

enthusiast_clean.loc[
    amg_gt_sports_mask,
    "generation"
] = "C190/R190"

In [88]:
amg_gt_4door_mask = (
    (enthusiast_clean["manufacturer"] == "Mercedes-Benz") &
    (enthusiast_clean["model_family"] == "AMG GT 4-Door")
)

enthusiast_clean.loc[
    amg_gt_4door_mask,
    "generation"
] = "X290"

In [89]:
mercedes_generation_check = (
    enthusiast_clean.loc[
        enthusiast_clean["manufacturer"] == "Mercedes-Benz"
    ]
    .groupby(
        ["model_family", "generation"],
        dropna=False
    )
    .size()
    .reset_index(name="listings")
)

mercedes_generation_check

,model_family,generation,listings
0,AMG GT 4-Door,X290,314
1,AMG GT Sports Car,C190/R190,297
2,C63,W204/C204,33
3,C63,W205/C205,309
4,E63,W211,14
5,E63,W212/S212,71
6,E63,W213/S213,135


In [90]:
mercedes_unmapped = enthusiast_clean.loc[
    (enthusiast_clean["manufacturer"] == "Mercedes-Benz") &
    (enthusiast_clean["generation"].isna()),
    [
        "source_row_id",
        "model_family",
        "year",
        "model"
    ]
]

print("Unmapped Mercedes-AMG rows:", len(mercedes_unmapped))

mercedes_unmapped.sort_values(
    ["model_family", "year"]
).head(50)

Unmapped Mercedes-AMG rows: 0


,source_row_id,model_family,year,model


### Porsche Generation Mapping

Porsche model families span multiple platform generations with substantially different market behavior.

Generation assignments are based primarily on model year. Where a model year can legitimately contain vehicles from two different generations, the record is assigned to a transition category rather than forcing an uncertain chassis classification.

In [91]:
porsche_911_mask = (
    (enthusiast_clean["manufacturer"] == "Porsche") &
    (enthusiast_clean["model_family"] == "911")
)

porsche_911_generation_map = [
    (1965, 1973, "Classic / Pre-G"),
    (1974, 1988, "G-Series"),
    (1989, 1993, "964"),
    (1994, 1998, "993"),
    (1999, 2004, "996"),
    (2005, 2011, "997"),
    (2013, 2019, "991"),
    (2020, 2023, "992")
]

for start_year, end_year, generation in porsche_911_generation_map:
    enthusiast_clean.loc[
        porsche_911_mask &
        enthusiast_clean["year"].between(start_year, end_year),
        "generation"
    ] = generation

In [92]:
enthusiast_clean.loc[
    porsche_911_mask &
    (enthusiast_clean["year"] == 2012),
    "generation"
] = "997/991 Transition"

In [93]:
cayman_mask = (
    (enthusiast_clean["manufacturer"] == "Porsche") &
    (enthusiast_clean["model_family"] == "Cayman / 718 Coupe")
)

enthusiast_clean.loc[
    cayman_mask &
    enthusiast_clean["year"].between(2006, 2012),
    "generation"
] = "987 Cayman"

enthusiast_clean.loc[
    cayman_mask &
    enthusiast_clean["year"].between(2013, 2016),
    "generation"
] = "981 Cayman"

enthusiast_clean.loc[
    cayman_mask &
    enthusiast_clean["year"].between(2017, 2023),
    "generation"
] = "718 / 982"

In [94]:
boxster_mask = (
    (enthusiast_clean["manufacturer"] == "Porsche") &
    (enthusiast_clean["model_family"] == "Boxster / 718 Roadster")
)

enthusiast_clean.loc[
    boxster_mask &
    enthusiast_clean["year"].between(1997, 2004),
    "generation"
] = "986 Boxster"

enthusiast_clean.loc[
    boxster_mask &
    enthusiast_clean["year"].between(2005, 2012),
    "generation"
] = "987 Boxster"

enthusiast_clean.loc[
    boxster_mask &
    enthusiast_clean["year"].between(2013, 2016),
    "generation"
] = "981 Boxster"

enthusiast_clean.loc[
    boxster_mask &
    enthusiast_clean["year"].between(2017, 2023),
    "generation"
] = "718 / 982"

In [95]:
porsche_generation_check = (
    enthusiast_clean.loc[
        enthusiast_clean["manufacturer"] == "Porsche"
    ]
    .groupby(
        ["model_family", "generation"],
        dropna=False
    )
    .size()
    .reset_index(name="listings")
)

porsche_generation_check

,model_family,generation,listings
0,911,964,34
1,911,991,1249
2,911,992,830
3,911,993,68
4,911,996,260
5,911,997,408
6,911,997/991 Transition,81
7,911,Classic / Pre-G,22
8,911,G-Series,71
9,Boxster / 718 Roadster,718 / 982,238


In [96]:
porsche_unmapped = enthusiast_clean.loc[
    (enthusiast_clean["manufacturer"] == "Porsche") &
    (enthusiast_clean["generation"].isna()),
    [
        "source_row_id",
        "model_family",
        "year",
        "model"
    ]
]

print("Unmapped Porsche rows:", len(porsche_unmapped))

porsche_unmapped.sort_values(
    ["model_family", "year"]
).head(50)

Unmapped Porsche rows: 0


,source_row_id,model_family,year,model


### Audi RS Generation Mapping

Audi RS models are mapped to platform generations using model year. Years that cannot be assigned confidently from model year alone are left unmapped for inspection rather than being forced into a generation.

In [97]:
rs3_mask = (
    (enthusiast_clean["manufacturer"] == "Audi") &
    (enthusiast_clean["model_family"] == "RS3")
)

enthusiast_clean.loc[
    rs3_mask &
    enthusiast_clean["year"].between(2017, 2020),
    "generation"
] = "8V"

enthusiast_clean.loc[
    rs3_mask &
    enthusiast_clean["year"].between(2022, 2023),
    "generation"
] = "8Y"

In [98]:
rs5_mask = (
    (enthusiast_clean["manufacturer"] == "Audi") &
    (enthusiast_clean["model_family"] == "RS5")
)

enthusiast_clean.loc[
    rs5_mask &
    enthusiast_clean["year"].between(2013, 2015),
    "generation"
] = "B8.5"

enthusiast_clean.loc[
    rs5_mask &
    enthusiast_clean["year"].between(2018, 2023),
    "generation"
] = "B9"

In [99]:
rs7_mask = (
    (enthusiast_clean["manufacturer"] == "Audi") &
    (enthusiast_clean["model_family"] == "RS7")
)

enthusiast_clean.loc[
    rs7_mask &
    enthusiast_clean["year"].between(2014, 2018),
    "generation"
] = "C7"

enthusiast_clean.loc[
    rs7_mask &
    enthusiast_clean["year"].between(2021, 2023),
    "generation"
] = "C8"

In [100]:
audi_generation_check = (
    enthusiast_clean.loc[
        enthusiast_clean["manufacturer"] == "Audi"
    ]
    .groupby(
        ["model_family", "generation"],
        dropna=False
    )
    .size()
    .reset_index(name="listings")
)

audi_generation_check

,model_family,generation,listings
0,RS3,8V,52
1,RS3,8Y,6
2,RS5,B8.5,53
3,RS5,B9,155
4,RS7,C7,71
5,RS7,C8,66


In [101]:
audi_unmapped = enthusiast_clean.loc[
    (enthusiast_clean["manufacturer"] == "Audi") &
    (enthusiast_clean["generation"].isna()),
    [
        "source_row_id",
        "model_family",
        "year",
        "model"
    ]
]

print("Unmapped Audi rows:", len(audi_unmapped))

audi_unmapped.sort_values(
    ["model_family", "year"]
)

Unmapped Audi rows: 0


,source_row_id,model_family,year,model


### Cadillac Blackwing Generation Mapping

The CT4-V Blackwing and CT5-V Blackwing listings in the dataset are limited to the 2022–2023 model years. Each is treated as the first Blackwing generation within its respective model family.

In [102]:
ct4_bw_mask = (
    (enthusiast_clean["manufacturer"] == "Cadillac") &
    (enthusiast_clean["model_family"] == "CT4-V Blackwing")
)

ct5_bw_mask = (
    (enthusiast_clean["manufacturer"] == "Cadillac") &
    (enthusiast_clean["model_family"] == "CT5-V Blackwing")
)

In [103]:
enthusiast_clean.loc[
    ct4_bw_mask,
    "generation"
] = "1st Gen Blackwing"

enthusiast_clean.loc[
    ct5_bw_mask,
    "generation"
] = "1st Gen Blackwing"

In [104]:
cadillac_generation_check = (
    enthusiast_clean.loc[
        enthusiast_clean["manufacturer"] == "Cadillac"
    ]
    .groupby(
        ["model_family", "generation"],
        dropna=False
    )
    .size()
    .reset_index(name="listings")
)

cadillac_generation_check

,model_family,generation,listings
0,CT4-V Blackwing,1st Gen Blackwing,81
1,CT5-V Blackwing,1st Gen Blackwing,57


## 8. Generation Mapping Validation

After generation mapping is completed for all included vehicle families, the full cleaned dataset is audited to ensure every listing has a generation assignment.

Transition-year categories are retained separately where model year alone cannot support a confident chassis assignment.

In [105]:
generation_missing = enthusiast_clean[
    enthusiast_clean["generation"].isna()
]

print(
    "Rows with missing generation:",
    len(generation_missing)
)

Rows with missing generation: 0


In [106]:
generation_summary = (
    enthusiast_clean
    .groupby(
        [
            "manufacturer",
            "model_family",
            "generation",
            "project_scope"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="listings")
    .sort_values(
        ["manufacturer", "model_family", "generation"]
    )
)

generation_summary

,manufacturer,model_family,generation,project_scope,listings
0,Audi,RS3,8V,Supporting,52
1,Audi,RS3,8Y,Supporting,6
2,Audi,RS5,B8.5,Core,53
3,Audi,RS5,B9,Core,155
4,Audi,RS7,C7,Supporting,71
5,Audi,RS7,C8,Supporting,66
6,BMW,M2,F87,Supporting,157
7,BMW,M3,E30,Core,5
8,BMW,M3,E36,Core,32
9,BMW,M3,E46,Core,87


In [107]:
enthusiast_clean["generation_quality_flag"] = "Mapped"

enthusiast_clean.loc[
    enthusiast_clean["generation"] == "997/991 Transition",
    "generation_quality_flag"
] = "Transition / Ambiguous"

In [108]:
enthusiast_clean[
    "generation_quality_flag"
].value_counts()

generation_quality_flag
Mapped                    10549
Transition / Ambiguous       81
Name: count, dtype: int64

In [109]:
enthusiast_clean.loc[
    enthusiast_clean["generation_quality_flag"]
    == "Transition / Ambiguous",
    [
        "manufacturer",
        "model_family",
        "year",
        "generation",
        "model"
    ]
].head(20)

,manufacturer,model_family,year,generation,model
614452,Porsche,911,2012,997/991 Transition,911 Carrera S
614468,Porsche,911,2012,997/991 Transition,911 Carrera 4 GTS
614479,Porsche,911,2012,997/991 Transition,911 Carrera 4 GTS
614500,Porsche,911,2012,997/991 Transition,911 Carrera S
614507,Porsche,911,2012,997/991 Transition,911 Carrera 4S
614519,Porsche,911,2012,997/991 Transition,911 Carrera 4S
614540,Porsche,911,2012,997/991 Transition,911 Turbo S
614577,Porsche,911,2012,997/991 Transition,911 Carrera
614724,Porsche,911,2012,997/991 Transition,911 Turbo
614926,Porsche,911,2012,997/991 Transition,911 Carrera S


In [110]:
analysis_df = enthusiast_clean[
    enthusiast_clean["generation_quality_flag"] == "Mapped"
]

In [111]:
print("Current rows:", len(enthusiast_clean))

print(
    "Duplicate rows beyond first occurrence:",
    enthusiast_clean.duplicated(
        subset=raw_columns,
        keep="first"
    ).sum()
)

print(
    "Generation flag total:",
    enthusiast_clean["generation_quality_flag"].value_counts().sum()
)

Current rows: 10630


NameError: name 'raw_columns' is not defined

In [112]:
raw_columns = df_raw.columns.tolist()

print("Original source columns:", len(raw_columns))

Original source columns: 20


In [113]:
print("Current rows:", len(enthusiast_clean))

print(
    "Duplicate rows beyond first occurrence:",
    enthusiast_clean.duplicated(
        subset=raw_columns,
        keep="first"
    ).sum()
)

print(
    "Generation flag total:",
    enthusiast_clean["generation_quality_flag"]
    .value_counts()
    .sum()
)

Current rows: 10630
Duplicate rows beyond first occurrence: 150
Generation flag total: 10630


In [114]:
enthusiast_clean = (
    enthusiast_clean
    .drop_duplicates(
        subset=raw_columns,
        keep="first"
    )
    .copy()
)

In [115]:
print("Rows after deduplication:", len(enthusiast_clean))

print(
    "Exact duplicate rows remaining:",
    enthusiast_clean.duplicated(
        subset=raw_columns,
        keep=False
    ).sum()
)

print(
    "Missing generations:",
    enthusiast_clean["generation"].isna().sum()
)

Rows after deduplication: 10480
Exact duplicate rows remaining: 0
Missing generations: 0


In [116]:
enthusiast_clean[
    "generation_quality_flag"
].value_counts()

generation_quality_flag
Mapped                    10400
Transition / Ambiguous       80
Name: count, dtype: int64

In [117]:
analysis_df = enthusiast_clean[
    enthusiast_clean["generation_quality_flag"] == "Mapped"
]

In [118]:
analysis_df = enthusiast_clean.loc[
    enthusiast_clean["generation_quality_flag"] == "Mapped"
].copy()

print("Analysis-ready mapped rows:", len(analysis_df))

Analysis-ready mapped rows: 10400


In [119]:
assert len(enthusiast_clean) == 10480, \
    "Unexpected row count — check whether deduplication was undone."

assert enthusiast_clean.duplicated(
    subset=raw_columns,
    keep=False
).sum() == 0, \
    "Exact duplicates remain in enthusiast_clean."

assert enthusiast_clean["generation"].isna().sum() == 0, \
    "Some vehicles are missing generation mappings."

assert (
    enthusiast_clean["generation_quality_flag"]
    .value_counts()
    .sum()
    == len(enthusiast_clean)
), "Generation quality flags do not reconcile."

print("Core cleaning checks passed.")

Core cleaning checks passed.


## 9. Vehicle History Normalization

Vehicle-history variables such as accident history, one-owner status, and personal-use status may influence enthusiast vehicle pricing.

The raw source fields are preserved, while standardized categorical versions are created for downstream SQL, statistical analysis, and dashboard reporting.

In [120]:
history_columns = [
    "accidents_or_damage",
    "one_owner",
    "personal_use_only"
]

for column in history_columns:
    print(f"\n--- {column} ---")
    print(
        enthusiast_clean[column]
        .value_counts(dropna=False)
        .sort_index()
    )


--- accidents_or_damage ---
accidents_or_damage
0.0    8203
1.0    1563
NaN     714
Name: count, dtype: int64

--- one_owner ---
one_owner
0.0    6220
1.0    3430
NaN     830
Name: count, dtype: int64

--- personal_use_only ---
personal_use_only
0.0    1819
1.0    7917
NaN     744
Name: count, dtype: int64


In [121]:
history_missing = (
    enthusiast_clean[history_columns]
    .isna()
    .sum()
    .to_frame("missing_rows")
)

history_missing["missing_pct"] = (
    history_missing["missing_rows"]
    / len(enthusiast_clean)
    * 100
).round(2)

history_missing

,missing_rows,missing_pct
accidents_or_damage,714,6.81
one_owner,830,7.92
personal_use_only,744,7.10


In [122]:
for column in history_columns:
    unexpected = enthusiast_clean.loc[
        enthusiast_clean[column].notna() &
        ~enthusiast_clean[column].isin([0, 1]),
        column
    ].unique()

    print(
        column,
        "unexpected values:",
        unexpected
    )

accidents_or_damage unexpected values: []
one_owner unexpected values: []
personal_use_only unexpected values: []


In [123]:
def normalize_binary_history(value, yes_label, no_label):
    if pd.isna(value):
        return "Unknown"

    if value == 1:
        return yes_label

    if value == 0:
        return no_label

    return "Unknown"

In [124]:
enthusiast_clean["accident_history"] = (
    enthusiast_clean["accidents_or_damage"]
    .apply(
        lambda x: normalize_binary_history(
            x,
            "Accident/Damage Reported",
            "No Accident/Damage Reported"
        )
    )
)

enthusiast_clean["ownership_history"] = (
    enthusiast_clean["one_owner"]
    .apply(
        lambda x: normalize_binary_history(
            x,
            "One Owner",
            "Multiple/Unknown Owners"
        )
    )
)

enthusiast_clean["usage_history"] = (
    enthusiast_clean["personal_use_only"]
    .apply(
        lambda x: normalize_binary_history(
            x,
            "Personal Use Only",
            "Not Personal Use Only"
        )
    )
)

In [125]:
for column in [
    "accident_history",
    "ownership_history",
    "usage_history"
]:
    print(f"\n--- {column} ---")
    print(
        enthusiast_clean[column]
        .value_counts(dropna=False)
    )


--- accident_history ---
accident_history
No Accident/Damage Reported    8203
Accident/Damage Reported       1563
Unknown                         714
Name: count, dtype: int64

--- ownership_history ---
ownership_history
Multiple/Unknown Owners    6220
One Owner                  3430
Unknown                     830
Name: count, dtype: int64

--- usage_history ---
usage_history
Personal Use Only        7917
Not Personal Use Only    1819
Unknown                   744
Name: count, dtype: int64


In [126]:
history_summary = pd.DataFrame({
    "accident_history": enthusiast_clean[
        "accident_history"
    ].value_counts(),

    "ownership_history": enthusiast_clean[
        "ownership_history"
    ].value_counts(),

    "usage_history": enthusiast_clean[
        "usage_history"
    ].value_counts()
})

history_summary

,accident_history,ownership_history,usage_history
Accident/Damage Reported,1563.0,NaN,NaN
Multiple/Unknown Owners,NaN,6220.0,NaN
No Accident/Damage Reported,8203.0,NaN,NaN
Not Personal Use Only,NaN,NaN,1819.0
One Owner,NaN,3430.0,NaN
Personal Use Only,NaN,NaN,7917.0
Unknown,714.0,830.0,744.0


In [127]:
def normalize_binary_field(value, yes_label, no_label):
    if pd.isna(value):
        return "Unknown"

    if value == 1:
        return yes_label

    if value == 0:
        return no_label

    return "Unknown"

In [128]:
enthusiast_clean["accident_history"] = (
    enthusiast_clean["accidents_or_damage"]
    .apply(
        lambda x: normalize_binary_field(
            x,
            "Accident/Damage Reported",
            "No Accident/Damage Reported"
        )
    )
)

enthusiast_clean["one_owner_status"] = (
    enthusiast_clean["one_owner"]
    .apply(
        lambda x: normalize_binary_field(
            x,
            "One Owner",
            "Not One Owner"
        )
    )
)

enthusiast_clean["personal_use_status"] = (
    enthusiast_clean["personal_use_only"]
    .apply(
        lambda x: normalize_binary_field(
            x,
            "Personal Use Only",
            "Not Personal Use Only"
        )
    )
)

In [129]:
for column in [
    "accident_history",
    "one_owner_status",
    "personal_use_status"
]:
    print(f"\n--- {column} ---")
    print(
        enthusiast_clean[column]
        .value_counts(dropna=False)
    )


--- accident_history ---
accident_history
No Accident/Damage Reported    8203
Accident/Damage Reported       1563
Unknown                         714
Name: count, dtype: int64

--- one_owner_status ---
one_owner_status
Not One Owner    6220
One Owner        3430
Unknown           830
Name: count, dtype: int64

--- personal_use_status ---
personal_use_status
Personal Use Only        7917
Not Personal Use Only    1819
Unknown                   744
Name: count, dtype: int64


## 10. Drivetrain and Fuel-Type Normalization

Drivetrain and fuel-type descriptions are inspected for inconsistent marketplace terminology before standardized categories are created.

Raw source values are preserved and normalized versions are added separately.

In [130]:
print("--- Drivetrain ---")
print(
    enthusiast_clean["drivetrain"]
    .value_counts(dropna=False)
)

print("\n--- Fuel Type ---")
print(
    enthusiast_clean["fuel_type"]
    .value_counts(dropna=False)
)

--- Drivetrain ---
drivetrain
Rear-wheel Drive     7257
All-wheel Drive      2641
NaN                   411
RWD                   121
AWD                    32
Four-wheel Drive       10
Front-wheel Drive       4
Rear-Wheel Drive        2
Unknown                 2
Name: count, dtype: int64

--- Fuel Type ---
fuel_type
Gasoline                         10195
NaN                                259
Hybrid                              18
Gasoline Fuel                        5
Gasoline/Mild Electric Hybrid        2
G                                    1
Name: count, dtype: int64


In [131]:
print(
    "Unique drivetrain values:",
    enthusiast_clean["drivetrain"].nunique(dropna=True)
)

print(
    "Missing drivetrain:",
    enthusiast_clean["drivetrain"].isna().sum()
)

print(
    "Unique fuel-type values:",
    enthusiast_clean["fuel_type"].nunique(dropna=True)
)

print(
    "Missing fuel type:",
    enthusiast_clean["fuel_type"].isna().sum()
)

Unique drivetrain values: 8
Missing drivetrain: 411
Unique fuel-type values: 5
Missing fuel type: 259


In [132]:
suspicious_drivetrain = enthusiast_clean.loc[
    enthusiast_clean["drivetrain"].isin([
        "Four-wheel Drive",
        "Front-wheel Drive",
        "Unknown"
    ]),
    [
        "source_row_id",
        "manufacturer",
        "model_family",
        "generation",
        "year",
        "model",
        "drivetrain"
    ]
]

suspicious_drivetrain.sort_values(
    ["drivetrain", "manufacturer", "model_family"]
)

,source_row_id,manufacturer,model_family,generation,year,model,drivetrain
614794,614794,Porsche,911,991,2016,911 Carrera,Four-wheel Drive
615983,615983,Porsche,911,996,1999,911 Carrera 4 Cabriolet,Four-wheel Drive
616067,616067,Porsche,911,991,2015,911 Turbo,Four-wheel Drive
616147,616147,Porsche,911,993,1994,911 Carrera 4,Four-wheel Drive
616184,616184,Porsche,911,997,2008,911 Carrera 4S Cabriolet,Four-wheel Drive
616213,616213,Porsche,911,997,2008,911 Carrera 4,Four-wheel Drive
616571,616571,Porsche,911,997/991 Transition,2012,911 Carrera 4S,Four-wheel Drive
617108,617108,Porsche,911,993,1994,911 Carrera 4,Four-wheel Drive
617148,617148,Porsche,911,997,2011,911 Carrera 4S,Four-wheel Drive
617368,617368,Porsche,911,991,2018,911 Carrera 4S,Four-wheel Drive


In [133]:
drivetrain_missing_by_model = (
    enthusiast_clean.loc[
        enthusiast_clean["drivetrain"].isna()
    ]
    .groupby("model_family")
    .size()
    .sort_values(ascending=False)
)

drivetrain_missing_by_model

model_family
911                       261
Corvette C8                23
Boxster / 718 Roadster     19
AMG GT 4-Door              18
M5                         14
Supra A90/A91              14
AMG GT Sports Car          11
Cayman / 718 Coupe         11
E63                         9
M4                          9
Corvette C7                 7
M3                          7
M2                          5
C63                         3
dtype: int64

In [134]:
unusual_fuel = enthusiast_clean.loc[
    enthusiast_clean["fuel_type"].notna() &
    (enthusiast_clean["fuel_type"] != "Gasoline"),
    [
        "source_row_id",
        "manufacturer",
        "model_family",
        "generation",
        "year",
        "model",
        "fuel_type"
    ]
]

unusual_fuel.sort_values(
    ["fuel_type", "manufacturer", "model_family"]
)

,source_row_id,manufacturer,model_family,generation,year,model,fuel_type
720105,720105,Toyota,Supra A90/A91,A90/A91,2021,Supra 2,G
41853,41853,BMW,M3,F80,2018,M3 Base,Gasoline Fuel
100864,100864,Chevrolet,Corvette C8,C8,2023,Corvette Stingray w/2LT,Gasoline Fuel
614687,614687,Porsche,911,997,2005,911 Carrera C2S,Gasoline Fuel
615611,615611,Porsche,911,991,2018,911 Carrera GTS,Gasoline Fuel
609387,609387,Porsche,Boxster / 718 Roadster,986 Boxster,2004,Boxster Base,Gasoline Fuel
22727,22727,Audi,RS7,C8,2021,RS 7 4.0T,Gasoline/Mild Electric Hybrid
517765,517765,Mercedes-Benz,AMG GT 4-Door,X290,2022,AMG GT 53 AMG GT 53,Gasoline/Mild Electric Hybrid
22680,22680,Audi,RS7,C8,2021,RS 7,Hybrid
22715,22715,Audi,RS7,C8,2021,RS 7 4.0T,Hybrid


In [135]:
def normalize_drivetrain(value):
    if pd.isna(value):
        return "Unknown"

    text = str(value).strip().upper()

    if text in [
        "REAR-WHEEL DRIVE",
        "REAR-WHEEL DRIVE".upper(),
        "RWD"
    ]:
        return "RWD"

    if text in [
        "ALL-WHEEL DRIVE",
        "AWD"
    ]:
        return "AWD"

    if text == "FOUR-WHEEL DRIVE":
        return "4WD"

    if text == "FRONT-WHEEL DRIVE":
        return "FWD"

    return "Unknown"

In [136]:
def normalize_drivetrain(value):
    if pd.isna(value):
        return "Unknown"

    text = str(value).strip().upper()

    if text in ["REAR-WHEEL DRIVE", "RWD"]:
        return "RWD"

    if text in ["ALL-WHEEL DRIVE", "AWD"]:
        return "AWD"

    if text == "FOUR-WHEEL DRIVE":
        return "4WD"

    if text == "FRONT-WHEEL DRIVE":
        return "FWD"

    return "Unknown"

In [137]:
enthusiast_clean["drivetrain_group"] = (
    enthusiast_clean["drivetrain"]
    .apply(normalize_drivetrain)
)

enthusiast_clean["drivetrain_group"].value_counts()

drivetrain_group
RWD        7380
AWD        2673
Unknown     413
4WD          10
FWD           4
Name: count, dtype: int64

In [138]:
enthusiast_clean["drivetrain_quality_flag"] = "Parsed"

enthusiast_clean.loc[
    enthusiast_clean["drivetrain"].isna(),
    "drivetrain_quality_flag"
] = "Missing"

enthusiast_clean.loc[
    enthusiast_clean["drivetrain"].eq("Unknown"),
    "drivetrain_quality_flag"
] = "Unknown Source Value"

enthusiast_clean.loc[
    enthusiast_clean["drivetrain_group"].isin(["FWD", "4WD"]),
    "drivetrain_quality_flag"
] = "Review"

In [139]:
enthusiast_clean[
    "drivetrain_quality_flag"
].value_counts()

drivetrain_quality_flag
Parsed                  10053
Missing                   411
Review                     14
Unknown Source Value        2
Name: count, dtype: int64

In [140]:
def normalize_drivetrain(value):
    if pd.isna(value):
        return "Unknown"

    text = str(value).strip().upper()

    if text in [
        "REAR-WHEEL DRIVE",
        "RWD"
    ]:
        return "RWD"

    if text in [
        "ALL-WHEEL DRIVE",
        "AWD",
        "FOUR-WHEEL DRIVE"
    ]:
        return "AWD"

    # FWD is inconsistent with the enthusiast vehicles
    # where it appears in this dataset.
    if text == "FRONT-WHEEL DRIVE":
        return "Unknown"

    return "Unknown"

In [141]:
enthusiast_clean["drivetrain_group"] = (
    enthusiast_clean["drivetrain"]
    .apply(normalize_drivetrain)
)

In [142]:
enthusiast_clean["drivetrain_quality_flag"] = "Parsed"

enthusiast_clean.loc[
    enthusiast_clean["drivetrain"].isna(),
    "drivetrain_quality_flag"
] = "Missing"

enthusiast_clean.loc[
    enthusiast_clean["drivetrain"].eq("Unknown"),
    "drivetrain_quality_flag"
] = "Unknown Source Value"

enthusiast_clean.loc[
    enthusiast_clean["drivetrain"].eq("Front-wheel Drive"),
    "drivetrain_quality_flag"
] = "Source inconsistency"

In [143]:
enthusiast_clean["drivetrain_group"].value_counts()

drivetrain_group
RWD        7380
AWD        2683
Unknown     417
Name: count, dtype: int64

In [144]:
enthusiast_clean["drivetrain_quality_flag"].value_counts()

drivetrain_quality_flag
Parsed                  10063
Missing                   411
Source inconsistency        4
Unknown Source Value        2
Name: count, dtype: int64

### Fuel-Type Normalization

Fuel-type descriptions contain a small number of alternate gasoline labels and electrified/hybrid classifications.

Because the raw source does not consistently distinguish full hybrid from mild-hybrid technology, electrified gasoline records are grouped conservatively into an `Electrified/Hybrid` category while the original `fuel_type` field is preserved.

In [145]:
def normalize_fuel_type(value):
    if pd.isna(value):
        return "Unknown"

    text = str(value).strip().upper()

    if text in [
        "GASOLINE",
        "GASOLINE FUEL",
        "G"
    ]:
        return "Gasoline"

    if text in [
        "HYBRID",
        "GASOLINE/MILD ELECTRIC HYBRID"
    ]:
        return "Electrified/Hybrid"

    return "Unknown"

In [146]:
enthusiast_clean["fuel_group"] = (
    enthusiast_clean["fuel_type"]
    .apply(normalize_fuel_type)
)

In [147]:
enthusiast_clean["fuel_quality_flag"] = "Parsed"

enthusiast_clean.loc[
    enthusiast_clean["fuel_type"].isna(),
    "fuel_quality_flag"
] = "Missing"

enthusiast_clean.loc[
    enthusiast_clean["fuel_group"] == "Unknown",
    "fuel_quality_flag"
] = "Unknown/Unparsed"

In [148]:
enthusiast_clean["fuel_group"].value_counts()

fuel_group
Gasoline              10201
Unknown                 259
Electrified/Hybrid       20
Name: count, dtype: int64

In [149]:
enthusiast_clean["fuel_quality_flag"].value_counts()

fuel_quality_flag
Parsed              10221
Unknown/Unparsed      259
Name: count, dtype: int64

In [150]:
assert enthusiast_clean["drivetrain_group"].value_counts().sum() == 10480
assert enthusiast_clean["fuel_group"].value_counts().sum() == 10480

assert (
    enthusiast_clean["drivetrain_quality_flag"]
    .value_counts()
    .sum()
    == 10480
)

assert (
    enthusiast_clean["fuel_quality_flag"]
    .value_counts()
    .sum()
    == 10480
)

print("Drivetrain and fuel normalization checks passed.")

Drivetrain and fuel normalization checks passed.


## 11. Engine and MPG Audit

Engine and fuel-economy fields are inspected to determine whether they contain enough consistent information to support downstream analysis.

The goal is not to normalize every possible marketplace description, but to retain fields that provide reliable analytical value.

In [151]:
print(
    "Unique engine descriptions:",
    enthusiast_clean["engine"].nunique(dropna=True)
)

print(
    "Missing engine:",
    enthusiast_clean["engine"].isna().sum()
)

engine_counts = (
    enthusiast_clean["engine"]
    .value_counts(dropna=False)
)

engine_counts.head(50)

Unique engine descriptions: 374
Missing engine: 305


engine
6.2L V8 16V GDI OHV                                                 1384
3.0L H6 24V GDI DOHC Twin Turbo                                      780
3.0L I6 24V GDI DOHC Twin Turbo                                      771
4.0L V8 32V GDI DOHC Twin Turbo                                      762
3.8L H6 24V GDI DOHC Twin Turbo                                      428
3.0L I6 24V GDI DOHC Turbo                                           400
4.0L H6 24V GDI DOHC                                                 337
NaN                                                                  305
4.4L V8 32V GDI DOHC Twin Turbo                                      294
3.8L H6 24V GDI DOHC                                                 277
6.2L V8 16V GDI OHV Supercharged                                     209
5.7L V8 16V MPFI OHV                                                 200
6.0L V8 16V MPFI OHV                                                 191
6.2L V8 16V MPFI OHV                        

In [152]:
engine_counts.tail(50)

engine
345.0HP 3.6L Flat 6 Cylinder Engine Gasoline Fuel                   1
3.2L H6 214hp 195ft. lbs.                                           1
320.0HP 3.6L Flat 6 Cylinder Engine Gasoline Fuel                   1
Turbocharged Gas Flat 6 3.8L/232                                    1
3.3L H6 24V DOHC Turbo                                              1
3.8L H6                                                             1
3.6L 6 Cyl. SMPI DOHC                                               1
New Arrival - complete gallery coming soon. Available immediatel    1
3.8 liter twin turbo DOHC boxer 6 cylinder, 560 HP                  1
415.0HP 3.8L Flat 6 Cylinder Engine Gasoline Fuel                   1
3.8L horizontally-opposed DOHC 24V 6-cyl engine-inc: dry sump lu    1
FI                                                                  1
6.3L H6 24V Turbo MPFI DOHC                                         1
3.8L 6-Cylinder BiTurbo                                             1
3.6L DOHC SMP

In [153]:
print(
    "Unique MPG descriptions:",
    enthusiast_clean["mpg"].nunique(dropna=True)
)

print(
    "Missing MPG:",
    enthusiast_clean["mpg"].isna().sum()
)

mpg_counts = (
    enthusiast_clean["mpg"]
    .value_counts(dropna=False)
)

mpg_counts.head(50)

Unique MPG descriptions: 114
Missing MPG: 2592


mpg
NaN      2592
15-20     518
18-26     476
15-27     375
17-29     361
16-25     355
16-23     348
17-25     346
17-26     339
15-22     281
15-21     277
18-25     242
17-24     242
19-24     242
16-22     227
18-24     218
20-29     218
18-23     212
19-27     196
20-28     183
14-20     167
19-28     166
17-23     150
16-24     146
20-26     138
22-30     129
21-28     105
15-18      94
20-30      79
19-25      65
19-26      62
15-23      61
17-22      60
13-19      57
16-21      54
24-31      51
18-27      47
0-0        46
15-24      46
25-32      45
13-20      39
16-26      34
15-19      34
15-25      32
20-25      26
13-21      20
16-27      19
14-19      16
23-32      16
18-0       13
Name: count, dtype: int64

In [154]:
mpg_counts.tail(50)

mpg
15-0       5
16-20      4
17-28      4
22-28      4
13-22      3
0-29       3
19-23      3
19-0       3
16-28      3
20-27      3
22-32      3
23         3
14-20.0    2
17-26.0    2
14-22      2
17-28.0    2
0-25       2
14-23      2
16-26.0    2
20-30.0    2
20-32      2
21-26      2
20-0       2
25-25      1
0-24       1
14-0       1
12-18.0    1
12-23      1
21-32      1
21-31      1
21-30      1
22-22      1
15-21.0    1
12-20      1
20-20      1
12-0       1
21-29      1
20         1
19-29      1
19-27.0    1
23-28      1
19         1
23-0.0     1
14-18      1
14         1
20-22      1
18-22      1
24         1
15-0.0     1
23-31      1
Name: count, dtype: int64

In [155]:
enthusiast_clean[
    [
        "manufacturer",
        "model_family",
        "generation",
        "year",
        "engine",
        "mpg"
    ]
].sample(
    30,
    random_state=42
)

,manufacturer,model_family,generation,year,engine,mpg
617413,Porsche,911,991,2015,3.4L H6 24V GDI DOHC,19-27
42251,BMW,M4,G82/G83,2022,3.0L I6 24V GDI DOHC Twin Turbo,NaN
41881,BMW,M3,E46,2005,3.2L I6 24V MPFI DOHC,16-23
43212,BMW,M5,F90,2019,4.4L V8 32V GDI DOHC Twin Turbo,15-21
42280,BMW,M4,F82/F83,2015,"3L I-6 gasoline direct injection, DOHC, variab...",17-26
517693,Mercedes-Benz,AMG GT 4-Door,X290,2020,3.0L I6 24V GDI DOHC Turbo,19-24
43391,BMW,M5,F90,2022,4.4L V8 32V GDI DOHC Twin Turbo,NaN
100441,Chevrolet,Corvette C7,C7,2016,6.2L V8 16V GDI OHV,17-29
608880,Porsche,Boxster / 718 Roadster,718 / 982,2017,2.5L H4 16V GDI DOHC Turbo,20-26
616959,Porsche,911,G-Series,1987,3.3L H6 12V SOHC Turbo,NaN


In [156]:
engine_mpg_missing = (
    enthusiast_clean
    .groupby("model_family")
    .agg(
        listings=("model_family", "size"),
        engine_missing=("engine", lambda x: x.isna().sum()),
        mpg_missing=("mpg", lambda x: x.isna().sum())
    )
)

engine_mpg_missing["engine_missing_pct"] = (
    engine_mpg_missing["engine_missing"]
    / engine_mpg_missing["listings"]
    * 100
).round(1)

engine_mpg_missing["mpg_missing_pct"] = (
    engine_mpg_missing["mpg_missing"]
    / engine_mpg_missing["listings"]
    * 100
).round(1)

engine_mpg_missing.sort_values(
    "mpg_missing_pct",
    ascending=False
)

,listings,engine_missing,mpg_missing,engine_missing_pct,mpg_missing_pct
model_family,,,,,
CT4-V Blackwing,80,0,73,0.0,91.2
CT5-V Blackwing,57,0,49,0.0,86.0
Corvette C6,515,0,412,0.0,80.0
Corvette C5,275,0,167,0.0,60.7
Corvette C8,914,22,476,2.4,52.1
RS7,137,0,46,0.0,33.6
M4,565,10,186,1.8,32.9
AMG GT 4-Door,314,18,78,5.7,24.8
RS3,58,0,12,0.0,20.7


### Engine Feature Extraction

Engine descriptions are highly detailed and contain hundreds of marketplace-specific variations. Rather than attempting to standardize every raw engine description, a small set of reliable analytical features is extracted conservatively.

The original `engine` field remains unchanged.

In [157]:
import re

In [158]:
def extract_displacement_l(value):
    if pd.isna(value):
        return np.nan

    text = str(value).upper()

    match = re.search(
        r"\b(\d+(?:\.\d+)?)\s*L\b",
        text
    )

    if match:
        displacement = float(match.group(1))

        # Guard against obviously nonsensical matches
        if 1.0 <= displacement <= 10.0:
            return displacement

    return np.nan


enthusiast_clean["engine_displacement_l"] = (
    enthusiast_clean["engine"]
    .apply(extract_displacement_l)
)

In [159]:
print(
    "Displacement successfully extracted:",
    enthusiast_clean["engine_displacement_l"].notna().sum()
)

print(
    "Displacement missing/unparsed:",
    enthusiast_clean["engine_displacement_l"].isna().sum()
)

enthusiast_clean[
    "engine_displacement_l"
].describe()

Displacement successfully extracted: 10129
Displacement missing/unparsed: 351


count    10129.000000
mean         4.177125
std          1.317061
min          2.000000
25%          3.000000
50%          3.800000
75%          5.700000
max          7.000000
Name: engine_displacement_l, dtype: float64

In [160]:
def extract_cylinders(value):
    if pd.isna(value):
        return np.nan

    text = str(value).upper()

    # Common compact engine notation:
    # V8, V6, I6, I4, H6, H4
    match = re.search(
        r"\b(?:V|I|H)[-\s]?(\d{1,2})\b",
        text
    )

    if match:
        cylinders = int(match.group(1))

        if cylinders in [3, 4, 5, 6, 8, 10, 12]:
            return cylinders

    # Descriptions such as "8 Cylinder Engine"
    match = re.search(
        r"\b(\d{1,2})\s*CYL(?:INDER)?",
        text
    )

    if match:
        cylinders = int(match.group(1))

        if cylinders in [3, 4, 5, 6, 8, 10, 12]:
            return cylinders

    # Porsche-style "Flat 6"
    match = re.search(
        r"\b(?:FLAT|BOXER|STRAIGHT)\s*[-]?\s*(\d{1,2})\b",
        text
    )

    if match:
        cylinders = int(match.group(1))

        if cylinders in [3, 4, 5, 6, 8, 10, 12]:
            return cylinders

    return np.nan


enthusiast_clean["engine_cylinders"] = (
    enthusiast_clean["engine"]
    .apply(extract_cylinders)
)

In [161]:
enthusiast_clean[
    "engine_cylinders"
].value_counts(
    dropna=False
).sort_index()

engine_cylinders
4.0      370
5.0       57
6.0     5286
8.0     4324
10.0      21
NaN      422
Name: count, dtype: int64

### MPG Parsing

The raw MPG field usually stores city and highway fuel economy as a string such as `17-26`. Because the field has substantial missingness and some invalid zero-valued records, numeric MPG fields are created only when both values are positive and plausibly formatted.

In [162]:
def parse_mpg(value):
    if pd.isna(value):
        return pd.Series(
            [np.nan, np.nan]
        )

    text = str(value).strip()

    match = re.fullmatch(
        r"(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)",
        text
    )

    if not match:
        return pd.Series(
            [np.nan, np.nan]
        )

    city = float(match.group(1))
    highway = float(match.group(2))

    # Reject source values such as 0-0 or 18-0
    if city <= 0 or highway <= 0:
        return pd.Series(
            [np.nan, np.nan]
        )

    return pd.Series(
        [city, highway]
    )

In [163]:
enthusiast_clean[
    ["city_mpg", "highway_mpg"]
] = enthusiast_clean["mpg"].apply(
    parse_mpg
)

In [164]:
enthusiast_clean["combined_mpg_simple"] = (
    enthusiast_clean[
        ["city_mpg", "highway_mpg"]
    ]
    .mean(axis=1)
)

In [165]:
feature_validation = pd.DataFrame({
    "available": [
        enthusiast_clean["engine_displacement_l"].notna().sum(),
        enthusiast_clean["engine_cylinders"].notna().sum(),
        enthusiast_clean["city_mpg"].notna().sum(),
        enthusiast_clean["highway_mpg"].notna().sum()
    ],
    "missing": [
        enthusiast_clean["engine_displacement_l"].isna().sum(),
        enthusiast_clean["engine_cylinders"].isna().sum(),
        enthusiast_clean["city_mpg"].isna().sum(),
        enthusiast_clean["highway_mpg"].isna().sum()
    ]
}, index=[
    "engine_displacement_l",
    "engine_cylinders",
    "city_mpg",
    "highway_mpg"
])

feature_validation["available_pct"] = (
    feature_validation["available"]
    / len(enthusiast_clean)
    * 100
).round(1)

feature_validation

,available,missing,available_pct
engine_displacement_l,10129,351,96.7
engine_cylinders,10058,422,96.0
city_mpg,7768,2712,74.1
highway_mpg,7768,2712,74.1


In [166]:
enthusiast_clean[
    [
        "model_family",
        "generation",
        "engine",
        "engine_displacement_l",
        "engine_cylinders",
        "mpg",
        "city_mpg",
        "highway_mpg"
    ]
].sample(
    25,
    random_state=42
)

,model_family,generation,engine,engine_displacement_l,engine_cylinders,mpg,city_mpg,highway_mpg
617413,911,991,3.4L H6 24V GDI DOHC,3.4,6.0,19-27,19.0,27.0
42251,M4,G82/G83,3.0L I6 24V GDI DOHC Twin Turbo,3.0,6.0,NaN,NaN,NaN
41881,M3,E46,3.2L I6 24V MPFI DOHC,3.2,6.0,16-23,16.0,23.0
43212,M5,F90,4.4L V8 32V GDI DOHC Twin Turbo,4.4,8.0,15-21,15.0,21.0
42280,M4,F82/F83,"3L I-6 gasoline direct injection, DOHC, variab...",3.0,6.0,17-26,17.0,26.0
517693,AMG GT 4-Door,X290,3.0L I6 24V GDI DOHC Turbo,3.0,6.0,19-24,19.0,24.0
43391,M5,F90,4.4L V8 32V GDI DOHC Twin Turbo,4.4,8.0,NaN,NaN,NaN
100441,Corvette C7,C7,6.2L V8 16V GDI OHV,6.2,8.0,17-29,17.0,29.0
608880,Boxster / 718 Roadster,718 / 982,2.5L H4 16V GDI DOHC Turbo,2.5,4.0,20-26,20.0,26.0
616959,911,G-Series,3.3L H6 12V SOHC Turbo,3.3,6.0,NaN,NaN,NaN


## 12. Final Analysis-Ready Schema

The final dataset retains the original source fields that are useful for analysis together with standardized model, generation, transmission, drivetrain, fuel, vehicle-history, and engine features.

Raw source columns are preserved where appropriate so normalized values remain auditable.

In [167]:
final_columns = [
    # Traceability / scope
    "source_row_id",
    "project_scope",

    # Vehicle identity
    "manufacturer",
    "model",
    "model_family",
    "generation",
    "generation_quality_flag",
    "year",

    # Market variables
    "price",
    "price_drop",
    "mileage",
    "mileage_quality_flag",

    # Transmission
    "transmission",
    "transmission_type",
    "transmission_group",
    "transmission_quality_flag",

    # Drivetrain
    "drivetrain",
    "drivetrain_group",
    "drivetrain_quality_flag",

    # Engine / fuel
    "engine",
    "engine_displacement_l",
    "engine_cylinders",
    "fuel_type",
    "fuel_group",
    "fuel_quality_flag",

    # MPG
    "mpg",
    "city_mpg",
    "highway_mpg",
    "combined_mpg_simple",

    # Vehicle history
    "accidents_or_damage",
    "accident_history",
    "one_owner",
    "one_owner_status",
    "personal_use_only",
    "personal_use_status",

    # Appearance
    "exterior_color",
    "interior_color",

    # Ratings
    "seller_rating",
    "driver_rating",
    "driver_reviews_num"
]

In [168]:
final_clean = (
    enthusiast_clean[final_columns]
    .copy()
    .reset_index(drop=True)
)

print("Final rows:", final_clean.shape[0])
print("Final columns:", final_clean.shape[1])

KeyError: "['mileage_quality_flag'] not in index"

In [169]:
missing_final_columns = [
    col for col in final_columns
    if col not in enthusiast_clean.columns
]

print("Missing columns:", missing_final_columns)

Missing columns: ['mileage_quality_flag']


In [170]:
print("Current enthusiast_clean columns:")
print(enthusiast_clean.columns.tolist())

Current enthusiast_clean columns:
['source_row_id', 'manufacturer', 'model', 'year', 'mileage', 'engine', 'transmission', 'drivetrain', 'fuel_type', 'mpg', 'exterior_color', 'interior_color', 'accidents_or_damage', 'one_owner', 'personal_use_only', 'seller_name', 'seller_rating', 'driver_rating', 'driver_reviews_num', 'price_drop', 'price', 'model_family', 'project_scope', 'generation', 'transmission_type', 'transmission_group', 'transmission_quality_flag', 'generation_quality_flag', 'accident_history', 'ownership_history', 'usage_history', 'one_owner_status', 'personal_use_status', 'drivetrain_group', 'drivetrain_quality_flag', 'fuel_group', 'fuel_quality_flag', 'engine_displacement_l', 'engine_cylinders', 'city_mpg', 'highway_mpg', 'combined_mpg_simple']


In [171]:
enthusiast_clean["mileage_quality_flag"] = "Valid"

enthusiast_clean.loc[
    enthusiast_clean["mileage"].isna(),
    "mileage_quality_flag"
] = "Missing"

In [172]:
enthusiast_clean["mileage_quality_flag"].value_counts()

mileage_quality_flag
Valid      10474
Missing        6
Name: count, dtype: int64

In [173]:
missing_final_columns = [
    col for col in final_columns
    if col not in enthusiast_clean.columns
]

print("Missing columns:", missing_final_columns)

Missing columns: []


In [174]:
final_clean = (
    enthusiast_clean[final_columns]
    .copy()
    .reset_index(drop=True)
)

print("Final rows:", final_clean.shape[0])
print("Final columns:", final_clean.shape[1])

Final rows: 10480
Final columns: 40


## 13. Final Dataset Validation

Before export, the completed analysis-ready dataset is subjected to final integrity checks.

Validation confirms:

- the expected row and column counts
- unique source-row identifiers
- removal of exact source-record duplicates
- complete model-family and generation assignments
- preservation of documented missing values
- consistency between missing values and quality flags
- exclusion of out-of-scope vehicles
- correct handling of known source inconsistencies

Quality-flagged records are retained rather than silently corrected or discarded.

In [176]:
print("Final shape:", final_clean.shape)

missing_final_columns = [
    col for col in final_columns
    if col not in final_clean.columns
]

unexpected_final_columns = [
    col for col in final_clean.columns
    if col not in final_columns
]

print("Missing final columns:", missing_final_columns)
print("Unexpected final columns:", unexpected_final_columns)

Final shape: (10480, 40)
Missing final columns: []
Unexpected final columns: []


In [177]:
exact_source_duplicates_remaining = (
    enthusiast_clean
    .duplicated(
        subset=raw_columns,
        keep="first"
    )
    .sum()
)

source_id_duplicates = (
    final_clean["source_row_id"]
    .duplicated()
    .sum()
)

print(
    "Exact source duplicates remaining:",
    exact_source_duplicates_remaining
)

print(
    "Duplicate source_row_id values:",
    source_id_duplicates
)

Exact source duplicates remaining: 0
Duplicate source_row_id values: 0


In [178]:
final_validation = pd.Series({
    "rows":
        len(final_clean),

    "columns":
        final_clean.shape[1],

    "unique_source_rows":
        final_clean["source_row_id"].nunique(),

    "exact_source_duplicates_remaining":
        exact_source_duplicates_remaining,

    "duplicate_source_row_ids":
        source_id_duplicates,

    "missing_model_family":
        final_clean["model_family"].isna().sum(),

    "missing_generation":
        final_clean["generation"].isna().sum(),

    "missing_project_scope":
        final_clean["project_scope"].isna().sum(),

    "missing_price":
        final_clean["price"].isna().sum(),

    "missing_mileage":
        final_clean["mileage"].isna().sum(),

    "excluded_scope_rows":
        (final_clean["project_scope"] == "Exclude").sum(),

    "ambiguous_generation":
        (
            final_clean["generation_quality_flag"]
            == "Transition / Ambiguous"
        ).sum(),

    "transmission_source_inconsistency":
        (
            final_clean["transmission_quality_flag"]
            == "Source inconsistency"
        ).sum(),

    "drivetrain_source_inconsistency":
        (
            final_clean["drivetrain_quality_flag"]
            == "Source inconsistency"
        ).sum(),

    "engine_displacement_available":
        final_clean["engine_displacement_l"]
        .notna()
        .sum(),

    "engine_cylinders_available":
        final_clean["engine_cylinders"]
        .notna()
        .sum(),

    "parsed_mpg_available":
        final_clean["city_mpg"]
        .notna()
        .sum()
})

final_validation

rows                                 10480
columns                                 40
unique_source_rows                   10480
exact_source_duplicates_remaining        0
duplicate_source_row_ids                 0
missing_model_family                     0
missing_generation                       0
missing_project_scope                    0
missing_price                            0
missing_mileage                          6
excluded_scope_rows                      0
ambiguous_generation                    80
transmission_source_inconsistency        0
drivetrain_source_inconsistency          4
engine_displacement_available        10129
engine_cylinders_available           10058
parsed_mpg_available                  7768
dtype: int64

In [179]:
print(
    "Missing mileage values:",
    final_clean["mileage"].isna().sum()
)

print(
    "Mileage rows flagged Missing:",
    (
        final_clean["mileage_quality_flag"]
        == "Missing"
    ).sum()
)

print(
    "Missing transmission values:",
    final_clean["transmission"].isna().sum()
)

print(
    "Transmission rows flagged Missing:",
    (
        final_clean["transmission_quality_flag"]
        == "Missing"
    ).sum()
)

print(
    "Missing drivetrain values:",
    final_clean["drivetrain"].isna().sum()
)

print(
    "Drivetrain rows flagged Missing:",
    (
        final_clean["drivetrain_quality_flag"]
        == "Missing"
    ).sum()
)

print(
    "Missing fuel values:",
    final_clean["fuel_type"].isna().sum()
)

print(
    "Fuel rows flagged Missing:",
    (
        final_clean["fuel_quality_flag"]
        == "Missing"
    ).sum()
)

Missing mileage values: 6
Mileage rows flagged Missing: 6
Missing transmission values: 306
Transmission rows flagged Missing: 306
Missing drivetrain values: 411
Drivetrain rows flagged Missing: 411
Missing fuel values: 259
Fuel rows flagged Missing: 0


In [180]:
c8_manual_remaining = (
    (
        final_clean["model_family"]
        == "Corvette C8"
    )
    &
    (
        final_clean["transmission_group"]
        == "Manual"
    )
).sum()

print(
    "C8 rows classified as Manual:",
    c8_manual_remaining
)

C8 rows classified as Manual: 15


In [181]:
c8_manual_conflict = (
    (enthusiast_clean["model_family"] == "Corvette C8") &
    (enthusiast_clean["transmission_type"] == "Manual")
)

print(
    "C8 manual source conflicts found:",
    c8_manual_conflict.sum()
)

C8 manual source conflicts found: 15


In [182]:
enthusiast_clean.loc[
    c8_manual_conflict,
    "transmission_type"
] = "Unknown/Ambiguous"

enthusiast_clean.loc[
    c8_manual_conflict,
    "transmission_group"
] = "Unknown"

enthusiast_clean.loc[
    c8_manual_conflict,
    "transmission_quality_flag"
] = "Source inconsistency"

In [183]:
print(
    "C8 rows still classified as Manual:",
    (
        (enthusiast_clean["model_family"] == "Corvette C8") &
        (enthusiast_clean["transmission_group"] == "Manual")
    ).sum()
)

print(
    "Transmission source inconsistencies:",
    (
        enthusiast_clean["transmission_quality_flag"]
        == "Source inconsistency"
    ).sum()
)

C8 rows still classified as Manual: 0
Transmission source inconsistencies: 15


In [184]:
enthusiast_clean["fuel_quality_flag"] = "Parsed"

# Non-missing values that could not be interpreted
enthusiast_clean.loc[
    enthusiast_clean["fuel_type"].notna() &
    (enthusiast_clean["fuel_group"] == "Unknown"),
    "fuel_quality_flag"
] = "Unknown/Unparsed"

# Actual missing source values
enthusiast_clean.loc[
    enthusiast_clean["fuel_type"].isna(),
    "fuel_quality_flag"
] = "Missing"

In [185]:
enthusiast_clean[
    "fuel_quality_flag"
].value_counts()

fuel_quality_flag
Parsed     10221
Missing      259
Name: count, dtype: int64

In [186]:
print(
    "Missing fuel values:",
    enthusiast_clean["fuel_type"].isna().sum()
)

print(
    "Fuel rows flagged Missing:",
    (
        enthusiast_clean["fuel_quality_flag"]
        == "Missing"
    ).sum()
)

Missing fuel values: 259
Fuel rows flagged Missing: 259


In [187]:
final_clean = (
    enthusiast_clean[final_columns]
    .copy()
    .reset_index(drop=True)
)

print("Final shape:", final_clean.shape)

Final shape: (10480, 40)


In [188]:
print(
    "C8 manual rows:",
    (
        (final_clean["model_family"] == "Corvette C8") &
        (final_clean["transmission_group"] == "Manual")
    ).sum()
)

print(
    "Transmission source inconsistencies:",
    (
        final_clean["transmission_quality_flag"]
        == "Source inconsistency"
    ).sum()
)

print(
    "Missing transmission values:",
    final_clean["transmission"].isna().sum()
)

print(
    "Transmission rows flagged Missing:",
    (
        final_clean["transmission_quality_flag"]
        == "Missing"
    ).sum()
)

print(
    "Missing fuel values:",
    final_clean["fuel_type"].isna().sum()
)

print(
    "Fuel rows flagged Missing:",
    (
        final_clean["fuel_quality_flag"]
        == "Missing"
    ).sum()
)

C8 manual rows: 0
Transmission source inconsistencies: 15
Missing transmission values: 306
Transmission rows flagged Missing: 306
Missing fuel values: 259
Fuel rows flagged Missing: 259


In [189]:
exact_source_duplicates_remaining = (
    enthusiast_clean
    .duplicated(
        subset=raw_columns,
        keep="first"
    )
    .sum()
)

source_id_duplicates = (
    final_clean["source_row_id"]
    .duplicated()
    .sum()
)

final_validation = pd.Series({
    "rows": len(final_clean),
    "columns": final_clean.shape[1],
    "unique_source_rows":
        final_clean["source_row_id"].nunique(),

    "exact_source_duplicates_remaining":
        exact_source_duplicates_remaining,

    "duplicate_source_row_ids":
        source_id_duplicates,

    "missing_model_family":
        final_clean["model_family"].isna().sum(),

    "missing_generation":
        final_clean["generation"].isna().sum(),

    "missing_project_scope":
        final_clean["project_scope"].isna().sum(),

    "missing_price":
        final_clean["price"].isna().sum(),

    "missing_mileage":
        final_clean["mileage"].isna().sum(),

    "excluded_scope_rows":
        (final_clean["project_scope"] == "Exclude").sum(),

    "ambiguous_generation":
        (
            final_clean["generation_quality_flag"]
            == "Transition / Ambiguous"
        ).sum(),

    "transmission_source_inconsistency":
        (
            final_clean["transmission_quality_flag"]
            == "Source inconsistency"
        ).sum(),

    "drivetrain_source_inconsistency":
        (
            final_clean["drivetrain_quality_flag"]
            == "Source inconsistency"
        ).sum(),

    "engine_displacement_available":
        final_clean["engine_displacement_l"].notna().sum(),

    "engine_cylinders_available":
        final_clean["engine_cylinders"].notna().sum(),

    "parsed_mpg_available":
        final_clean["city_mpg"].notna().sum()
})

final_validation

rows                                 10480
columns                                 40
unique_source_rows                   10480
exact_source_duplicates_remaining        0
duplicate_source_row_ids                 0
missing_model_family                     0
missing_generation                       0
missing_project_scope                    0
missing_price                            0
missing_mileage                          6
excluded_scope_rows                      0
ambiguous_generation                    80
transmission_source_inconsistency       15
drivetrain_source_inconsistency          4
engine_displacement_available        10129
engine_cylinders_available           10058
parsed_mpg_available                  7768
dtype: int64

In [190]:
assert len(final_clean) == 10480
assert final_clean.shape[1] == 40

assert final_clean["source_row_id"].nunique() == 10480

assert exact_source_duplicates_remaining == 0
assert source_id_duplicates == 0

assert final_clean["model_family"].isna().sum() == 0
assert final_clean["generation"].isna().sum() == 0
assert final_clean["project_scope"].isna().sum() == 0
assert final_clean["price"].isna().sum() == 0

assert (
    final_clean["project_scope"]
    .isin(["Core", "Supporting"])
    .all()
)

assert (
    final_clean["mileage"].isna().sum()
    ==
    (
        final_clean["mileage_quality_flag"]
        == "Missing"
    ).sum()
)

assert (
    final_clean["transmission"].isna().sum()
    ==
    (
        final_clean["transmission_quality_flag"]
        == "Missing"
    ).sum()
)

assert (
    final_clean["drivetrain"].isna().sum()
    ==
    (
        final_clean["drivetrain_quality_flag"]
        == "Missing"
    ).sum()
)

assert (
    final_clean["fuel_type"].isna().sum()
    ==
    (
        final_clean["fuel_quality_flag"]
        == "Missing"
    ).sum()
)

assert (
    (
        (final_clean["model_family"] == "Corvette C8") &
        (final_clean["transmission_group"] == "Manual")
    ).sum()
    == 0
)

assert (
    final_clean["drivetrain_group"]
    == "FWD"
).sum() == 0

assert (
    final_clean["generation_quality_flag"]
    .notna()
    .all()
)

assert (
    final_clean["transmission_quality_flag"]
    .notna()
    .all()
)

assert (
    final_clean["drivetrain_quality_flag"]
    .notna()
    .all()
)

assert (
    final_clean["fuel_quality_flag"]
    .notna()
    .all()
)

print("FINAL DATASET VALIDATION PASSED.")

FINAL DATASET VALIDATION PASSED.


In [191]:
analysis_df = (
    final_clean.loc[
        final_clean["generation_quality_flag"] == "Mapped"
    ]
    .copy()
    .reset_index(drop=True)
)

print("Master cleaned dataset:", len(final_clean))
print("Generation-analysis dataset:", len(analysis_df))
print(
    "Transition rows excluded:",
    len(final_clean) - len(analysis_df)
)

Master cleaned dataset: 10480
Generation-analysis dataset: 10400
Transition rows excluded: 80


In [192]:
PROCESSED_PATH.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_PATH = (
    PROCESSED_PATH /
    "apex_enthusiast_cars_clean.csv"
)

final_clean.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Exported to:", OUTPUT_PATH)

print(
    "File size:",
    round(
        OUTPUT_PATH.stat().st_size / (1024 ** 2),
        2
    ),
    "MB"
)

Exported to: ../data/processed/apex_enthusiast_cars_clean.csv
File size: 3.42 MB


In [193]:
export_check = pd.read_csv(OUTPUT_PATH)

print("Exported rows:", export_check.shape[0])
print("Exported columns:", export_check.shape[1])
print(
    "Unique source rows:",
    export_check["source_row_id"].nunique()
)

Exported rows: 10480
Exported columns: 40
Unique source rows: 10480


In [194]:
assert export_check.shape == (10480, 40)

assert (
    export_check["source_row_id"].nunique()
    == 10480
)

assert export_check["model_family"].isna().sum() == 0
assert export_check["generation"].isna().sum() == 0
assert export_check["price"].isna().sum() == 0

print("EXPORT VERIFICATION PASSED.")

EXPORT VERIFICATION PASSED.
